# Servidor MCP (Model Context Protocol)

Servidor FastMCP con tres herramientas, dos recursos y dos prompts. Ejecute las celdas en orden.

Al final, el mismo código queda consolidado en `servidor_mcp.py` para clientes MCP (stdio).

## Creación del servidor FastMCP

In [1]:
# FastMCP is the high-level framework for the Model Context Protocol.
# It handles transport negotiation, JSON-Schema generation, and the protocol
# wire format so this tutorial can focus on defining tools, resources, and prompts.
from mcp.server.fastmcp import FastMCP
# instructions is included in the server manifest sent to MCP clients.
# Hosts may surface it as context when the model decides which server to query.
mcp_server = FastMCP(
    name = "agents-tutorial-server",
    instructions = (
        "Educational MCP server — unit conversion, arithmetic, and weather tools. "
        "Part of the Agents Tutorials series (TA07)."
    ),
)

c:\Users\mesab\taller2-git-dashboard\.venv\Lib\site-packages\pydantic_settings\sources\utils.py:47: IncompleteFieldDefinitionWarning: Field 'lifespan' has an incomplete definition: its annotation contains an unresolved forward reference, so settings sources may fail to correctly resolve its value. Call `model_rebuild()` on the model where the field is defined, once all the referenced types are defined.
  warnings.warn(


In [2]:
print(f"Server:       {mcp_server.name!r}")
print(f"Instructions: {mcp_server.instructions!r}")
print(f"Transport:    stdio (default — spawned as a subprocess by MCP hosts)")
print(f"Protocol:     Model Context Protocol (MCP) via FastMCP")
print(f"Capabilities: tools + resources + prompts")

Server:       'agents-tutorial-server'
Instructions: 'Educational MCP server — unit conversion, arithmetic, and weather tools. Part of the Agents Tutorials series (TA07).'
Transport:    stdio (default — spawned as a subprocess by MCP hosts)
Protocol:     Model Context Protocol (MCP) via FastMCP
Capabilities: tools + resources + prompts


## Definición de herramientas

In [3]:
from typing import Annotated, Literal
from pydantic import Field
import ast, operator
import requests
import json

### Herramienta de conversión de unidades

In [4]:
# Each key is a (from_unit, to_unit) tuple; the value is a conversion lambda.
# Lookup is O(1) and adding a new unit pair requires only a new entry here —
# no changes to the tool logic are needed.
CONVERSIONS = {
    ("km", "miles"):           lambda x: x * 0.621371,
    ("miles", "km"):           lambda x: x * 1.60934,
    ("celsius", "fahrenheit"): lambda x: x * 9 / 5 + 32,
    ("fahrenheit", "celsius"): lambda x: (x - 32) * 5 / 9,
    ("kg", "lbs"):             lambda x: x * 2.20462,
    ("lbs", "kg"):             lambda x: x * 0.453592,
    ("meters", "feet"):        lambda x: x * 3.28084,
    ("feet", "meters"):        lambda x: x * 0.3048,
    ("liters", "gallons"):     lambda x: x * 0.264172,
    ("gallons", "liters"):     lambda x: x * 3.78541,
}

In [5]:
@mcp_server.tool()
def convert_units(
    value:     Annotated[float, Field(description = "Numeric value to convert")],
    from_unit: Annotated[str,   Field(description = "Source unit: km, miles, celsius, fahrenheit, kg, lbs, meters, feet, liters, gallons")],
    to_unit:   Annotated[str,   Field(description = "Target unit: km, miles, celsius, fahrenheit, kg, lbs, meters, feet, liters, gallons")],
) -> str:
    
    """Convert a numeric value between supported units of measurement.

    Supports ten bidirectional pairs: km↔miles, celsius↔fahrenheit, kg↔lbs,
    meters↔feet, liters↔gallons. Returns a formatted equality string on success
    or a descriptive error message when the pair is not in CONVERSIONS, letting
    the model relay a meaningful response instead of an opaque failure.

    Args:
        value:     The numeric quantity to convert.
        from_unit: Source unit name (case-insensitive, whitespace stripped).
        to_unit:   Target unit name (same normalisation as from_unit).

    Returns:
        "{value} {from_unit} = {result:.4f} {to_unit}" on success, or an
        unsupported-pair message listing valid combinations.
    """

    # Normalise before lookup so "KM", " km " and "km" all resolve correctly.
    key = (from_unit.lower().strip(), to_unit.lower().strip())

    if key in CONVERSIONS:
        return f"{value} {from_unit} = {CONVERSIONS[key](value):.4f} {to_unit}"

    return (
        f"Conversion from {from_unit!r} to {to_unit!r} is not supported. "
        f"Valid pairs: km/miles, celsius/fahrenheit, kg/lbs, meters/feet, liters/gallons."
    )

In [6]:
print("Tool 1/3 registered: convert_units")
print("  Required params: value (float), from_unit (str), to_unit (str)")
print("  All three params are always relevant — no conditional fields.")
print("  Schema source: Annotated[type, Field(description=...)] on each parameter")

Tool 1/3 registered: convert_units
  Required params: value (float), from_unit (str), to_unit (str)
  All three params are always relevant — no conditional fields.
  Schema source: Annotated[type, Field(description=...)] on each parameter


### Herramienta de cálculo aritmético seguro

In [7]:
# Whitelist of AST node types mapped to their safe Python operator equivalents.
# Any node NOT listed here — function calls, attribute access, name lookups,
# imports — causes _safe_eval to raise ValueError, blocking code injection
# at the parse stage before any execution occurs.
ALLOWED_OPS = {
    ast.Add:  operator.add,
    ast.Sub:  operator.sub,
    ast.Mult: operator.mul,
    ast.Div:  operator.truediv,
    ast.Pow:  operator.pow,
    ast.Mod:  operator.mod,
    ast.USub: operator.neg,
}

In [8]:
def _safe_eval(node):

    """Recursively evaluate an AST node using only whitelisted arithmetic operators.

    Walks the syntax tree produced by ast.parse() and applies the corresponding
    Python operator for each node. Accepts only numeric constants, binary
    operations (e.g. +, *, **), and unary negation. Any other node type —
    function calls, attribute access, name lookups — raises ValueError before
    any code executes, blocking injection attempts at the AST level.

    Args:
        node: An ast.expr node, typically tree.body from
              ast.parse(expression, mode="eval").

    Returns:
        The numeric result (int or float) of the expression.

    Raises:
        ValueError: If the node type is not in ALLOWED_OPS or is not a numeric
                    constant. The caller wraps this in a user-facing message.
    """
    
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value

    elif isinstance(node, ast.BinOp) and type(node.op) in ALLOWED_OPS:
        return ALLOWED_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))

    elif isinstance(node, ast.UnaryOp) and type(node.op) in ALLOWED_OPS:
        return ALLOWED_OPS[type(node.op)](_safe_eval(node.operand))

    # Anything else — Call, Attribute, Name, Import — is rejected.
    raise ValueError(f"Unsupported node type: {type(node).__name__}")

In [9]:
@mcp_server.tool()
def calculate(
    expression: Annotated[str, Field(description = "Arithmetic expression using +, -, *, /, **, % and numbers")],
) -> str:
    
    """Evaluate a mathematical expression safely without using eval().

    Uses ast.parse() to convert the expression to a syntax tree, then walks it
    with _safe_eval(), which only processes nodes listed in ALLOWED_OPS. This
    prevents code injection: __import__('os').system('...') parses to a Call
    node, which _safe_eval rejects before any OS command executes.

    Args:
        expression: A string containing a mathematical expression, e.g. "2 ** 10"
                    or "15 * 7 + 3". Supports +, -, *, /, **, % and unary minus.

    Returns:
        A string of the form "{expression} = {result}".

    Raises:
        ValueError: If the expression contains unsupported syntax or node types.
    """
    
    try:
        tree = ast.parse(expression, mode = "eval")
        result = _safe_eval(tree.body) 
        return f"{expression} = {result}"

    except ValueError as e:
        raise ValueError(f"Invalid expression {expression!r}: {e}")

    except SyntaxError:
        raise ValueError(f"Syntax error in expression: {expression!r}")

In [10]:
print("Tool 2/3 registered: calculate")
print("  Required params: expression (str)")
print("  Security: AST-based evaluation — Call/Attribute/Name nodes rejected before execution")

Tool 2/3 registered: calculate
  Required params: expression (str)
  Security: AST-based evaluation — Call/Attribute/Name nodes rejected before execution


### Herramienta de consulta del clima

In [11]:
# get_weather fetches live conditions from wttr.in, a public weather API that
# requires no API key and accepts city names directly. The timeout=5 guard
# prevents a slow network from stalling the tool execution indefinitely.

@mcp_server.tool()
def get_weather(
    city: Annotated[str, Field(description="City name to get current weather for (e.g. 'London', 'Buenos Aires')")],
) -> str:
    """Return the current weather for a city using the wttr.in public API.

    Fetches real-time conditions from wttr.in, which requires no API key and
    accepts city names in any language. On network failure or an unrecognised
    city, returns a descriptive error string rather than raising an exception,
    so the model always receives a usable result to relay.

    Args:
        city: Name of the city to query.

    Returns:
        A summary string with temperature (°C), weather description, and
        humidity, or an error message if the HTTP request fails.
    """
    try:
        response = requests.get(
            f"https://wttr.in/{city}?format=j1",
            timeout=5,
        )
        response.raise_for_status()
        current  = response.json()["current_condition"][0]
        temp_c   = current["temp_C"]
        desc     = current["weatherDesc"][0]["value"].lower()
        humidity = current["humidity"]
        return f"Weather in {city}: {temp_c}°C, {desc}, humidity {humidity}%"
    except requests.RequestException as e:
        return f"Could not fetch weather for {city!r}: {e}"

In [12]:
print("Tool 3/3 registered: get_weather")
print("  Required params: city (str)")
print("  Data source: wttr.in public API — no API key required")
print("  Error handling: RequestException is caught and returned as a string")

Tool 3/3 registered: get_weather
  Required params: city (str)
  Data source: wttr.in public API — no API key required
  Error handling: RequestException is caught and returned as a string


## Granularidad en el diseño de herramientas

In [13]:
# A minimal server used only to register convert_or_calculate for the
# fine-grained vs coarse-grained comparison below. Not wired to any transport.
demo_server = FastMCP("granularity-demo")
# Counter-example: one tool that handles two distinct operations.
# The schema has conditional fields — the model must infer from 'operation'
# which of the remaining parameters are relevant, increasing the chance of
# irrelevant fields being filled in or the schema being misunderstood.
@demo_server.tool()
def convert_or_calculate(
    operation:  Annotated[Literal["convert", "calculate"], Field(description = "Operation to perform")],
    value:      Annotated[float, Field(description = "Value to convert — ignored when operation='calculate'")] = 0,
    from_unit:  Annotated[str,   Field(description = "Source unit  — ignored when operation='calculate'")] = "",
    to_unit:    Annotated[str,   Field(description = "Target unit  — ignored when operation='calculate'")] = "",
    expression: Annotated[str,   Field(description = "Expression   — ignored when operation='convert'")] = "",
) -> str:
    
    """Coarse-grained counter-example: a single tool handling two operations.

    Illustrates why fine-grained tools (one tool per operation) are preferred
    in MCP server design. The schema contains conditional fields — parameters
    the model must decide to ignore based on the value of 'operation'. Compare
    the JSON schema printed in the next cell with that of convert_units and
    calculate: each fine-grained tool has only required, always-relevant params.
    """

    if operation == "convert":
        return f"{value} {from_unit} = ??? {to_unit}"
    
    return f"result: {expression}"

In [14]:
# list_tools() returns the JSON Schema the server would send to MCP clients.
# Fetching both before the print cell avoids an extra round-trip mid-display.
coarse = await demo_server.list_tools()
fine   = await mcp_server.list_tools()
ct = coarse[0]

print("====== COARSE TOOL (one tool, two operations) ======")
print(f"  {ct.name}")
print(f"  Total params:    {len(ct.inputSchema['properties'])}")
print(f"  Required:        {ct.inputSchema.get('required', [])}")
print(f"  Problem:         the model must infer which params are relevant")
print(f"                   based on the value of 'operation' — ambiguous schema")

print()
print("====== FINE-GRAINED TOOLS (one tool per operation) ======")
for ft in fine:
    req   = ft.inputSchema.get("required", [])
    total = len(ft.inputSchema["properties"])
    print(f"  {ft.name:16s}  params={total}, required={req}")

print()
print("Principle: each tool should do exactly one thing.")
print("  Fine-grained schemas have no conditional fields, no params the model")
print("  must decide to ignore — every listed param is always relevant.")

====== COARSE TOOL (one tool, two operations) ======
  convert_or_calculate
  Total params:    5
  Required:        ['operation']
  Problem:         the model must infer which params are relevant
                   based on the value of 'operation' — ambiguous schema

====== FINE-GRAINED TOOLS (one tool per operation) ======
  convert_units     params=3, required=['value', 'from_unit', 'to_unit']
  calculate         params=1, required=['expression']
  get_weather       params=1, required=['city']

Principle: each tool should do exactly one thing.
  Fine-grained schemas have no conditional fields, no params the model
  must decide to ignore — every listed param is always relevant.


### Esquema JSON completo

In [15]:
tools = await mcp_server.list_tools()

print(f"====== JSON SCHEMA sent to MCP clients ({len(tools)} tools) ======\n")
for t in tools:
    print(f"-- {t.name} --")
    print(json.dumps(t.inputSchema, indent=2, ensure_ascii=False))
    print()

====== JSON SCHEMA sent to MCP clients (3 tools) ======

-- convert_units --
{
  "properties": {
    "value": {
      "description": "Numeric value to convert",
      "title": "Value",
      "type": "number"
    },
    "from_unit": {
      "description": "Source unit: km, miles, celsius, fahrenheit, kg, lbs, meters, feet, liters, gallons",
      "title": "From Unit",
      "type": "string"
    },
    "to_unit": {
      "description": "Target unit: km, miles, celsius, fahrenheit, kg, lbs, meters, feet, liters, gallons",
      "title": "To Unit",
      "type": "string"
    }
  },
  "required": [
    "value",
    "from_unit",
    "to_unit"
  ],
  "title": "convert_unitsArguments",
  "type": "object"
}

-- calculate --
{
  "properties": {
    "expression": {
      "description": "Arithmetic expression using +, -, *, /, **, % and numbers",
      "title": "Expression",
      "type": "string"
    }
  },
  "required": [
    "expression"
  ],
  "title": "calculateArguments",
  "type": "object"


## Definición de recursos

In [16]:
@mcp_server.resource("config://server/info")
def get_server_info() -> str:

    """Metadata and capabilities of this MCP server."""

    return json.dumps({
        "name":    "agentes-tutorial-server",
        "version": "1.0.0",
        "tools":     ["convert_units", "calculate", "get_weather"],
        "resources": ["config://server/info", "units://reference/{category}"],
        "prompts":   ["conversion_assistant", "math_tutor"],
    }, indent = 2, ensure_ascii = False)

In [17]:
print("Resource 1/2 registered: config://server/info  (static)")
print("  URI:   fixed — every read returns the same server metadata")
print("  Type:  application/json via json.dumps()")

Resource 1/2 registered: config://server/info  (static)
  URI:   fixed — every read returns the same server metadata
  Type:  application/json via json.dumps()


In [18]:
UNITS_REFERENCE = {
    "length":      {"km": "kilometers", "miles": "miles", "meters": "meters", "feet": "feet"},
    "temperature": {"celsius": "Celsius (C)", "fahrenheit": "Fahrenheit (F)"},
    "mass":        {"kg": "kilograms", "lbs": "pounds"},
    "volume":      {"liters": "liters", "gallons": "gallons"},
}
@mcp_server.resource("units://reference/{category}")
def get_units_reference(category: str) -> str:

    """Unit reference table for a given category (length, temperature, mass, volume)."""

    data = UNITS_REFERENCE.get(category.lower())

    if data is None:
        return json.dumps(
            {"error": f"Category {category!r} not found", "available": list(UNITS_REFERENCE.keys())},
            ensure_ascii = False,
        )
    
    return json.dumps({category: data}, indent = 2, ensure_ascii = False)

In [19]:
print("Resource 2/2 registered: units://reference/{category}  (template)")
print("  URI pattern: one definition covers all of:")

for cat in UNITS_REFERENCE:
    print(f"    units://reference/{cat}")
print("  The {{category}} segment is passed as the `category` argument at read time.")

Resource 2/2 registered: units://reference/{category}  (template)
  URI pattern: one definition covers all of:
    units://reference/length
    units://reference/temperature
    units://reference/mass
    units://reference/volume
  The {{category}} segment is passed as the `category` argument at read time.


In [20]:
# list_resources() returns fixed URIs; list_resource_templates() returns
# URI patterns with variable segments (e.g. {category}) resolved at read time.
resources = await mcp_server.list_resources()
templates = await mcp_server.list_resource_templates()
print(f"Static resources  ({len(resources)}):  clients read these with a fixed URI")
for r in resources:
    print(f"  {r.uri}")

print(f"\nResource templates ({len(templates)}):  URI contains a variable segment")
for t in templates:
    print(f"  {t.uriTemplate}")
    print(f"    → {t.description}")

Static resources  (1):  clients read these with a fixed URI
  config://server/info

Resource templates (1):  URI contains a variable segment
  units://reference/{category}
    → Unit reference table for a given category (length, temperature, mass, volume).


## Definición de prompts

In [21]:
@mcp_server.prompt()
def conversion_assistant(unit_system: str = "metric") -> str:

    """System prompt that configures the assistant as a unit conversion expert."""

    system = "metric" if unit_system == "metric" else "imperial"
    
    return (
        f"You are a unit conversion expert. The user prefers the {system} system. "
        f"Always show the step-by-step procedure and round results to 4 decimal places."
    )

In [22]:
print("Prompt 1/2 registered: conversion_assistant")
print("  Argument:  unit_system (str, default='metric')")
print()
print("  Rendered with unit_system='metric':")
print(f"    {conversion_assistant()}")
print()
print("  Rendered with unit_system='imperial':")
print(f"    {conversion_assistant('imperial')}")

Prompt 1/2 registered: conversion_assistant
  Argument:  unit_system (str, default='metric')

  Rendered with unit_system='metric':
    You are a unit conversion expert. The user prefers the metric system. Always show the step-by-step procedure and round results to 4 decimal places.

  Rendered with unit_system='imperial':
    You are a unit conversion expert. The user prefers the imperial system. Always show the step-by-step procedure and round results to 4 decimal places.


In [23]:
@mcp_server.prompt()
def math_tutor(level: str = "intermediate") -> str:

    """System prompt that configures the assistant as a math tutor at a given level."""

    depth = {
        "beginner":     "Use simple examples and avoid formal notation.",
        "intermediate": "Use standard notation and explain the reasoning at each step.",
        "advanced":     "Assume algebra knowledge and use precise mathematical notation.",
    }
    
    return f"You are a math tutor. Student level: {level}. {depth.get(level, depth['intermediate'])}"

In [24]:
print("Prompt 2/2 registered: math_tutor")
print("  Argument:  level (str, default='intermediate')")
print()
for lvl in ["beginner", "intermediate", "advanced"]:
    print(f"  level={lvl!r}:")
    print(f"    {math_tutor(lvl)}")

Prompt 2/2 registered: math_tutor
  Argument:  level (str, default='intermediate')

  level='beginner':
    You are a math tutor. Student level: beginner. Use simple examples and avoid formal notation.
  level='intermediate':
    You are a math tutor. Student level: intermediate. Use standard notation and explain the reasoning at each step.
  level='advanced':
    You are a math tutor. Student level: advanced. Assume algebra knowledge and use precise mathematical notation.


In [25]:
prompts = await mcp_server.list_prompts()
print(f"====== Registered prompts ({len(prompts)}) ======\n")
for p in prompts:
    args = [(a.name, "optional" if not a.required else "required") for a in (p.arguments or [])]
    print(f"  {p.name}")
    print(f"    {p.description}")
    print(f"    arguments: {args}")

====== Registered prompts (2) ======

  conversion_assistant
    System prompt that configures the assistant as a unit conversion expert.
    arguments: [('unit_system', 'optional')]
  math_tutor
    System prompt that configures the assistant as a math tutor at a given level.
    arguments: [('level', 'optional')]


## Prueba local del servidor

In [26]:
print("====== Testing tools: correct sintax ======\n")

r = await mcp_server.call_tool("convert_units", {"value": 100, "from_unit": "km", "to_unit": "miles"})
print(f"  convert_units(100 km → miles):          {r[0].text}")

r = await mcp_server.call_tool("convert_units", {"value": 37, "from_unit": "celsius", "to_unit": "fahrenheit"})
print(f"  convert_units(37 celsius → fahrenheit): {r[0].text}")

r = await mcp_server.call_tool("calculate", {"expression": "15 * 7 + 3"})
print(f"  calculate('15 * 7 + 3'):                {r[0].text}")

r = await mcp_server.call_tool("calculate", {"expression": "2 ** 10"})
print(f"  calculate('2 ** 10'):                   {r[0].text}")

# get_weather hits the live wttr.in API — actual values will vary.
r = await mcp_server.call_tool("get_weather", {"city": "Bogota"})
print(f"  get_weather('Bogota'):                  {r[0].text}")

r = await mcp_server.call_tool("get_weather", {"city": "Tokyo"})
print(f"  get_weather('Tokyo'):                   {r[0].text}")

====== Testing tools: correct sintax ======

  convert_units(100 km → miles):          100.0 km = 62.1371 miles
  convert_units(37 celsius → fahrenheit): 37.0 celsius = 98.6000 fahrenheit
  calculate('15 * 7 + 3'):                15 * 7 + 3 = 108
  calculate('2 ** 10'):                   2 ** 10 = 1024
  get_weather('Bogota'):                  Weather in Bogota: 10°C, partly cloudy , humidity 93%
  get_weather('Tokyo'):                   Weather in Tokyo: 26°C, patchy rain nearby, humidity 75%


In [27]:
print("====== Testing tools: error handling ======\n")

# Unsupported unit pair — returns a descriptive error string, not an exception.
# The MCP client receives a valid ToolMessage the agent can relay to the user.
r = await mcp_server.call_tool("convert_units", {"value": 1, "from_unit": "km", "to_unit": "parsecs"})
print(f"  convert_units(km → parsecs): {r[0].text}")

# Code-injection attempt — ast.parse produces a Call node which _safe_eval
# rejects before any execution. The exception becomes a McpError in the response.
print()
try:
    await mcp_server.call_tool("calculate", {"expression": "__import__('os').system('ls')"})
except Exception as e:
    print(f"  calculate(injection attempt): {type(e).__name__} raised")
    print(f"    (the Call node was blocked at the AST level before any code ran)")

====== Testing tools: error handling ======

  convert_units(km → parsecs): Conversion from 'km' to 'parsecs' is not supported. Valid pairs: km/miles, celsius/fahrenheit, kg/lbs, meters/feet, liters/gallons.

  calculate(injection attempt): ToolError raised
    (the Call node was blocked at the AST level before any code ran)


In [28]:
print("====== Testing resources ======\n")

# Static resource — fixed URI, always returns the same content
config = await mcp_server.read_resource("config://server/info")
print("config://server/info (static):")
print(config[0].content)

# Template resource — URI variable is resolved at read time
temp = await mcp_server.read_resource("units://reference/temperature")
print("\nunits://reference/temperature:")
print(temp[0].content)

# Template resource — category that does not exist
missing = await mcp_server.read_resource("units://reference/speed")
print("\nunits://reference/speed (category not found):")
print(missing[0].content)

====== Testing resources ======

config://server/info (static):
{
  "name": "agentes-tutorial-server",
  "version": "1.0.0",
  "tools": [
    "convert_units",
    "calculate",
    "get_weather"
  ],
  "resources": [
    "config://server/info",
    "units://reference/{category}"
  ],
  "prompts": [
    "conversion_assistant",
    "math_tutor"
  ]
}

units://reference/temperature:
{
  "temperature": {
    "celsius": "Celsius (C)",
    "fahrenheit": "Fahrenheit (F)"
  }
}

units://reference/speed (category not found):
{"error": "Category 'speed' not found", "available": ["length", "temperature", "mass", "volume"]}


In [29]:
print("====== Testing prompts ======\n")
print("Prompts return PromptMessage objects that a client injects into a conversation.\n")

# get_prompt returns a GetPromptResult with a 'messages' list.
# Each message has a role ('user' or 'assistant') and a TextContent payload.

p = await mcp_server.get_prompt("conversion_assistant", {"unit_system": "imperial"})
print(f"conversion_assistant(unit_system='imperial'):")
print(f"  role:    {p.messages[0].role}")
print(f"  content: {p.messages[0].content.text}")

print()
p = await mcp_server.get_prompt("math_tutor", {"level": "beginner"})
print(f"math_tutor(level='beginner'):")
print(f"  role:    {p.messages[0].role}")
print(f"  content: {p.messages[0].content.text}")

====== Testing prompts ======

Prompts return PromptMessage objects that a client injects into a conversation.

conversion_assistant(unit_system='imperial'):
  role:    user
  content: You are a unit conversion expert. The user prefers the imperial system. Always show the step-by-step procedure and round results to 4 decimal places.

math_tutor(level='beginner'):
  role:    user
  content: You are a math tutor. Student level: beginner. Use simple examples and avoid formal notation.


## Servidor independiente

El código consolidado está en `servidor_mcp.py`.

En Windows (stdio, para clientes MCP):

```
python servidor_mcp.py
```

Inspector interactivo:

```
mcp dev servidor_mcp.py
```

El modo stdio no imprime nada en la terminal: espera un cliente por stdin/stdout.